## Feature extraction

**Objective:** Build and validate feature extraction on a single pilot subject before 
scaling to the full cohort. Covers all feature families: band power, PLI, coherence, PLV, 
and Kuramoto. Built incrementally across sessions; each family is validated on the pilot 
subject before the next is added.

**Inputs:**
- Pilot subject's preprocessed epochs (restEC, heog_off), from `data/derivatives/<subject_id>/`
- QC dict from `run_autoreject` (n_epochs, autoreject_extreme, consensus/n_interpolate)
- Bands matched to Chang et al. (2025): delta 2-4 Hz, theta 4-8 Hz, alpha 8-13 Hz, 
  beta 13-30 Hz, gamma 30-45 Hz 

**Assumptions (from modelling_decisions.md):**
- Mean aggregation across retained epochs, per feature, per subject 
- Raw, untransformed feature output for power; log-transform is modelling-stage only 
- PLI is the primary connectivity metric; coherence and PLV are secondary 
- QC dict passed in as-is, not re-derived 
- Failed subjects get a flagged row (NaN features, reason column), never a dropped None
- Naming: `{channel}_{band}_power`, `{channel1}_{channel2}_{band}_{metric}` 
- Full feature bank computed once; primary arm's curated pool is a column subset applied 
  at modelling time, not a separate extraction path

**Progress:**
- [X] Band power
- [ ] PLI
- [ ] Coherence, PLV
- [ ] Kuramoto

In [1]:
# imports and setup
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import importlib
import src.preprocessing
importlib.reload(src.preprocessing)
from src.preprocessing import *

In [2]:
# Load pilot subject (same subject from '02_preprocessing_pilot.ipynb') 

data_dir = find_repo_root() / "data"

subject_id = 'sub-87999321'
condition = 'restEC'
variant = 'heog_off'

epochs_path = data_dir / f'derivatives_{variant}' / subject_id / f'{subject_id}_{condition}-epo.fif'
epochs = mne.read_epochs(epochs_path)

print(epochs)

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
<EpochsFIF | 23 events (all good), 0 – 4.998 s (baseline off), ~13.6 MiB, data loaded,
 '1': 23>


In [3]:
# Load QC 
qc_log = pd.read_csv(data_dir / 'batch_results_log_full_cohort.csv')
print(qc_log.columns.tolist())
print(qc_log.head())

['subject_id', 'condition', 'heog_variant', 'status', 'n_epochs_before', 'n_epochs_after', 'output_path', 'autoreject_consensus', 'autoreject_n_interpolate', 'autoreject_extreme', 'heog_n_candidates', 'heog_n_valid', 'heog_correction_applied', 'error']
     subject_id condition heog_variant status  n_epochs_before  \
0  sub-87999321    restEC     heog_off     ok             24.0   
1  sub-87999321    restEO     heog_off     ok             24.0   
2  sub-87999321    restEC      heog_on     ok             24.0   
3  sub-87999321    restEO      heog_on     ok             24.0   
4  sub-88049537    restEC     heog_off     ok             24.0   

   n_epochs_after                                        output_path  \
0            23.0  /Users/romyweinstock/eeg-rtms-response-predict...   
1            22.0  /Users/romyweinstock/eeg-rtms-response-predict...   
2            23.0  /Users/romyweinstock/eeg-rtms-response-predict...   
3            22.0  /Users/romyweinstock/eeg-rtms-response-pred

In [4]:
# Filter the combined QC log down to this subject's row for the given condition and HEOG variant. 
qc_row = qc_log[
    (qc_log['subject_id'] == subject_id) &
    (qc_log['condition'] == condition) &
    (qc_log['heog_variant'] == variant)
]

# Fail loudly if the filter doesn't return exactly one row
assert len(qc_row) == 1, f"expected exactly 1 QC row, got {len(qc_row)}"

# Convert to a plain dict so it can be passed into extract_subject_features
qc_info = qc_row.iloc[0].to_dict()

print(qc_info)

{'subject_id': 'sub-87999321', 'condition': 'restEC', 'heog_variant': 'heog_off', 'status': 'ok', 'n_epochs_before': 24.0, 'n_epochs_after': 23.0, 'output_path': '/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif', 'autoreject_consensus': 0.2, 'autoreject_n_interpolate': 4.0, 'autoreject_extreme': False, 'heog_n_candidates': 4.0, 'heog_n_valid': 0.0, 'heog_correction_applied': False, 'error': nan}


In [5]:
# Compute band power
spectrum = epochs.compute_psd(method='welch')
print(spectrum)

# Get frequencies and power values
freqs = spectrum.freqs
power_data_array = spectrum.get_data()

# Confirm channel alignment: epochs.ch_names includes non-EEG channels
# (Erbs, Mass, Status, VEOG, HEOG) - 31 total - while spectrum only computed
# PSD for the 26 EEG channels. Use spectrum.ch_names, not epochs.ch_names,
# whenever pairing channel names with power values.
print(len(epochs.ch_names), len(spectrum.ch_names))
print(power_data_array.shape)  # (23 epochs, 26 channels, 1025 freqs)

# Band boundaries,  half-open intervals (lower inclusive, upper exclusive)
# to avoid double-counting a bin that lands exactly on a boundary
bands = {
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

# For each band: mask the relevant frequency bins, sum across them,
# then mean across epochs -> one value per channel
band_power = {}

for band_name, (low, high) in bands.items():
    mask = (freqs >= low) & (freqs < high)
    band_slice = power_data_array[:, :, mask]      # (23, 26, n_freqs_in_band)
    band_sum = np.sum(band_slice, axis=2)            # (23, 26)
    band_mean = np.mean(band_sum, axis=0) * 1e12  # V^2/Hz -> uV^2/Hz

    for channel, value in zip(spectrum.ch_names, band_mean):
        band_power[f"{channel}_{band_name}_power"] = value
# Confirm no NaNs slipped through - a NaN here would silently propagate into
# every downstream step (dataframe row, modelling matrix) without erroring
assert not any(np.isnan(v) for v in band_power.values()), "NaN found in band_power"

print(len(band_power))              # expect 130 (26 channels x 5 bands)
print(list(band_power.items())[:5]) # spot-check a few entries

# Pull out the alpha values specifically for O1, O2, Pz (posterior) and Fp1, Fp2 (frontal), and compare them 
posterior = ['O1', 'O2', 'Pz']
frontal = ['Fp1', 'Fp2']

for ch in posterior + frontal:
    print(ch, band_power[f"{ch}_alpha_power"])

Effective window size : 4.096 (s)
<Power Spectrum (from Epochs, welch method) | 23 epochs × 26 channels × 1025 freqs, 0.0-250.0 Hz>
31 26
(23, 26, 1025)
130
[('Fp1_delta_power', np.float64(35.41828852559409)), ('Fp2_delta_power', np.float64(34.988788065319014)), ('F7_delta_power', np.float64(28.24902379025797)), ('F3_delta_power', np.float64(33.13220605971701)), ('Fz_delta_power', np.float64(40.28291528903178))]
O1 208.11167873107232
O2 187.32865650971894
Pz 170.79886995121103
Fp1 110.49959001661445
Fp2 113.23930917617187


In [6]:
## Check pilot subject using built functions from 'fratures.py'

# Import funcitons 
from src.features import (load_subject_epochs, get_subject_qc, compute_band_power)

# Define needed parameters
data_dir = find_repo_root() / "data"
qc_log = pd.read_csv(data_dir / 'batch_results_log_full_cohort.csv')
bands = {
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

epochs = load_subject_epochs(subject_id = 'sub-87999321',condition = 'restEC', variant = 'heog_off', data_dir = data_dir)
qc_info = get_subject_qc(subject_id = 'sub-87999321',condition = 'restEC', variant = 'heog_off', qc_log = qc_log)
band_power = compute_band_power(epochs = epochs, bands = bands)

#checks
# Column count check
print(len(band_power))  # expect 130

# Posterior vs frontal alpha - should match the notebook version
posterior = ['O1', 'O2', 'Pz']
frontal = ['Fp1', 'Fp2']

for ch in posterior + frontal:
    print(ch, band_power[f"{ch}_alpha_power"])

# Exact-match spot check against the validated notebook value
print(band_power['Fp1_delta_power'])  # expect ~35.418288525594...

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
130
O1 208.11167873107232
O2 187.32865650971894
Pz 170.79886995121103
Fp1 110.49959001661445
Fp2 113.23930917617187
35.41828852559409


In [7]:
## Check orchestrator function (successful run)
from src.features import (extract_subject_features)

results = extract_subject_features(
    subject_id = 'sub-87999321',
    condition = 'restEC',
     variant = 'heog_off', 
     data_dir = data_dir,
     qc_log = qc_log,
     bands = bands
     )
results 

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)


{'subject_id': 'sub-87999321',
 'condition': 'restEC',
 'variant': 'heog_off',
 'reason': 'ok',
 'heog_variant': 'heog_off',
 'preprocessing_status': 'ok',
 'n_epochs_before': 24.0,
 'n_epochs_after': 23.0,
 'output_path': '/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif',
 'autoreject_consensus': 0.2,
 'autoreject_n_interpolate': 4.0,
 'autoreject_extreme': False,
 'heog_n_candidates': 4.0,
 'heog_n_valid': 0.0,
 'heog_correction_applied': False,
 'preprocessing_error': nan,
 'Fp1_delta_power': np.float64(35.41828852559409),
 'Fp2_delta_power': np.float64(34.988788065319014),
 'F7_delta_power': np.float64(28.24902379025797),
 'F3_delta_power': np.float64(33.13220605971701),
 'Fz_delta_power': np.float64(40.28291528903178),
 'F4_delta_power': np.float64(31.59585452670465),
 'F8_delta_power': np.float64(16.74755844345798),
 'FC3_delta_power': np.float64(32.14540771782294),
 'FCz_delta_power': np.float64(45.678578112546

In [8]:
## Check orchestrator function (failed run - madeup subject)

results = extract_subject_features(
    subject_id = 'sub-87999328',
    condition = 'restEC',
     variant = 'heog_off', 
     data_dir = data_dir,
     qc_log = qc_log,
     bands = bands
     )
results

{'subject_id': 'sub-87999328',
 'condition': 'restEC',
 'variant': 'heog_off',
 'reason': 'File does not exist: "/Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999328/sub-87999328_restEC-epo.fif"'}

In [ ]:
# Build one real success row and one real failure row, then check that
# concatenating them into a dataframe produces clean NaN-filling for the
# columns the failure row never computed

success_row = extract_subject_features(
    subject_id='sub-87999321', condition='restEC', variant='heog_off',
    data_dir=data_dir, qc_log=qc_log, bands=bands
)

failure_row = extract_subject_features(
    subject_id='sub-87999328', condition='restEC', variant='heog_off',
    data_dir=data_dir, qc_log=qc_log, bands=bands
)

test_df = pd.DataFrame([success_row, failure_row])

print(test_df.shape)
print(test_df[['subject_id', 'condition', 'variant', 'reason']])

# Check a QC column and a feature column specifically - both should be
# NaN for the failure row, real values for the success row
print(test_df[['n_epochs_after', 'Fp1_delta_power']])

Reading /Users/romyweinstock/eeg-rtms-response-prediction/data/derivatives_heog_off/sub-87999321/sub-87999321_restEC-epo.fif ...
    Found the data of interest:
        t =       0.00 ...    4998.00 ms
        0 CTF compensation matrices available
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Effective window size : 4.096 (s)
(2, 146)
     subject_id condition   variant  \
0  sub-87999321    restEC  heog_off   
1  sub-87999328    restEC  heog_off   

                                              reason  
0                                                 ok  
1  File does not exist: "/Users/romyweinstock/eeg...  
   n_epochs_after  Fp1_delta_power
0            23.0        35.418289
1             NaN              NaN
subject_id        0
condition         0
variant           0
reason            0
heog_variant      1
                 ..
P4_gamma_power    1
P8_gamma_power    1
O1_gamma_power    1
Oz_gamma_power    1
O2_gamma_power 

In [ ]:
#validate how many Nas exist across the 2-row test dataframe.
print(test_df.isna().sum())

subject_id        0
condition         0
variant           0
reason            0
heog_variant      1
                 ..
P4_gamma_power    1
P8_gamma_power    1
O1_gamma_power    1
Oz_gamma_power    1
O2_gamma_power    1
Length: 146, dtype: int64


## Interpretation - band power

Band power extraction validated on the pilot subject (sub-87999321, restEC, heog_off).

**Checklist results (Decision 5):**
- Column count: 130 (26 channels x 5 bands), exact match
- No NaNs: confirmed via assert, both in the notebook derivation and the
  refactored `compute_band_power` function
- Units: raw PSD output is in V^2/Hz (MNE's native unit), not uV^2/Hz as
  reported in the literature. Converted via *1e12. Caught before it reached
  any stored feature - the unconverted value would have been off by 12
  orders of magnitude.
- Physiological plausibility: posterior alpha (O1, O2, Pz: ~171-208 uV^2/Hz)
  visibly exceeds frontal alpha (Fp1, Fp2: ~110-113 uV^2/Hz), consistent
  with expected resting-EEG topography.

**Decisions made this session (to be added to modelling_decisions.md):**
- PSD method: Welch, not MNE's multitaper default. Chang et al. (2025)
  states "FFT" without specifying segmentation; Roelofs et al. (2021)'s
  similarly-described method is Welch by construction, and Stolz et al.
  (2023) - a separate TDBRAIN-based rTMS study - names Welch explicitly.
  Welch's default window size (4.096s) was accepted as MNE's default, not
  independently checked against segment-length choices in the cited papers.

**Bugs caught during build (not shipped):**
- `epochs.ch_names` (31, includes EOG/Erbs/Mass/Status) vs
  `spectrum.ch_names` (26, EEG only) - using the former would have
  silently mislabeled every power value.
- Band mask boundaries initially double-counted bins landing exactly on a
  boundary (`<=` on both sides); fixed to half-open intervals.
- QC log's own `status`/`error` columns collided with the orchestrator's
  `reason` field on merge; renamed to `preprocessing_status` /
  `preprocessing_error` to disambiguate pipeline stage.

**Refactor validation:** `compute_band_power` in `src/features.py`
reproduces the notebook-derived values exactly (spot-checked
Fp1_delta_power: 35.41828852559409, identical to 15 decimal places).

**Orchestrator (`extract_subject_features`):** tested against one real
success case and one deliberate failure case (nonexistent subject ID).
Confirmed `pd.DataFrame([...])` correctly NaN-fills missing QC/feature
columns for failed rows with no custom handling needed.


**Progress:**
- [x] Band power
- [ ] PLI
- [ ] Coherence, PLV
- [ ] Kuramoto